In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%capture
!pip install TA-Lib

# FX-BOT Unified Pipeline


In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr
import talib
import xgboost as xgb
import joblib
import os
import time
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
xgb.set_config(verbosity=0)


### 1. Data Repository


In [6]:
import numpy as np

import pandas as pd
import yfinance as yf

from tqdm import tqdm

import time
import os
import pandas_datareader as pdr

# https://finance.yahoo.com/markets/currencies/

CURRENCY_PAIRS = [
    'EURUSD=X', 'JPY=X', 'GBPUSD=X', 'AUDUSD=X', 'NZDUSD=X', 'EURJPY=X', 'GBPJPY=X',
    'EURGBP=X', 'EURCAD=X', 'EURSEK=X', 'EURCHF=X', 'EURHUF=X', 'CNY=X', 'HKD=X',
    'SGD=X', 'INR=X', 'MXN=X', 'PHP=X', 'IDR=X', 'THB=X', 'MYR=X', 'ZAR=X', 'RUB=X'
]


class DataRepository:
    ticker_df: pd.DataFrame
    indexes_df: pd.DataFrame
    macro_df: pd.DataFrame

    # min_date: str
    ALL_TICKERS: list[str] = CURRENCY_PAIRS  # 23 Currency Pairs

    def __init__(self):
        self.ticker_df = None
        self.indexes_df = None
        self.macro_df = None

    def _get_growth_df(self, df: pd.DataFrame, prefix: str) -> pd.DataFrame:
        '''Help function to produce a df with growth columns'''
        for i in [1, 4, 7, 10, 15]:
            df['growth_' + prefix + '_' + str(i) + 'h'] = df['Close'] / df['Close'].shift(i)
            GROWTH_KEYS = [k for k in df.keys() if k.startswith('growth')]
        return df[GROWTH_KEYS]

    def _fetch_index_with_fallback(self, yf_symbol, name, period='max'):
        '''Fetch index data with yfinance'''
        data = None
        # Try Yahoo Finance API
        try:
            ticker_obj = yf.Ticker(yf_symbol)
            data = ticker_obj.history(period=period, interval="1h")
            if not data.empty:
                print(f"yfinance SUCCESS: Got {len(data)} rows for {name}")
                data.index = pd.to_datetime(data.index)
            else:
                print(f"yfinance returned no data for {name}")
        except Exception as e:
            print(f"yfinance error for {name}: {e}")
        # If no data, create empty DataFrame with proper structure
        if data is None or data.empty:
            print(f"No data available for {name} from any source, creating empty DataFrame")
            data = pd.DataFrame(columns=['Open', 'High', 'Low', 'Close', 'Volume'])
            data.index = pd.DatetimeIndex([])
        # Sleep to avoid overloading API
        time.sleep(1)
        return data

    def fetch(self, period=None):
        '''Fetch all data from APIs'''

        print('Fetching Tickers info from YFinance')
        self.fetch_tickers(period=period)
        print('Fetching Indexes info from YFinance')
        self.fetch_indexes(period=period)
        print('Fetching Macro info from FRED (Pandas_datareader)')
        self.fetch_macro(min_date=None)

    def fetch_tickers(self, period='max'):
        '''Fetch Tickers data from yfinance API with Stooq fallback for REAL data'''

        print(f'Going download data for this tickers: {self.ALL_TICKERS}')
        tq = tqdm(self.ALL_TICKERS)

        for i, ticker in enumerate(tq):
            tq.set_description(ticker)
            historyPrices = None

            try:
                ticker_obj = yf.Ticker(ticker)
                historyPrices = ticker_obj.history(period=period, interval="1h")

                if not historyPrices.empty:
                    print(f"yfinance SUCCESS: Got {len(historyPrices)} rows for {ticker}")
                else:
                    print(f"yfinance returned no data for {ticker}")

            except Exception as e:
                print(f"yfinance error for {ticker}: {e}")

            # Skip if no data from either source
            if historyPrices is None or historyPrices.empty:
                print(f"No data available for {ticker} from YFinance, skipping...")
                time.sleep(1)
                continue

            # Ensure index is datetime
            historyPrices.index = pd.to_datetime(historyPrices.index)

            # generate features for historical prices, and what we want to predict
            historyPrices['Ticker'] = ticker
            historyPrices['Year'] = historyPrices.index.year
            historyPrices['Month'] = historyPrices.index.month
            historyPrices['Weekday'] = historyPrices.index.weekday
            historyPrices['DateTime'] = historyPrices.index
            historyPrices['DateTime'] = historyPrices.index
            historyPrices['Date'] = historyPrices.index.date

            # historical returns
            for i in [1, 4, 7, 10, 15]:
                historyPrices['growth_' + str(i) + 'h'] = historyPrices['Close'] / historyPrices['Close'].shift(i)
            # future returns
            historyPrices['growth_future_1h'] = historyPrices['Close'].shift(-1) / historyPrices['Close']
            historyPrices['growth_future_4h'] = historyPrices['Close'].shift(-4) / historyPrices['Close']

            # Technical indicators
            # SimpleMovingAverage 10 days and 20 days
            historyPrices['SMA10'] = historyPrices['Close'].rolling(10).mean()
            historyPrices['SMA20'] = historyPrices['Close'].rolling(20).mean()
            historyPrices['growing_moving_average'] = np.where(historyPrices['SMA10'] > historyPrices['SMA20'], 1, 0)
            historyPrices['high_minus_low_relative'] = (historyPrices['High'] - historyPrices['Low']) / historyPrices[
                'Close']

            # 30d rolling volatility : https://ycharts.com/glossary/terms/rolling_vol_30
            historyPrices['volatility'] = historyPrices['growth_1h'].rolling(30).std() * np.sqrt(252 * 24)

            # what we want to predict
            historyPrices['is_positive_growth_1h_future'] = np.where(historyPrices['growth_future_1h'] > 1, 1, 0)
            historyPrices['is_positive_growth_4h_future'] = np.where(historyPrices['growth_future_4h'] > 1, 1, 0)

            # sleep 1 sec between downloads - not to overload the API server
            time.sleep(1)

            if self.ticker_df is None:
                self.ticker_df = historyPrices
            else:
                self.ticker_df = pd.concat([self.ticker_df, historyPrices], ignore_index=True)

    def fetch_indexes(self, period='max'):
        '''Fetch Indexes data from yfinance '''
        # Define index mappings: (yfinance_symbol, name)
        index_mappings = [
            ("^GDAXI", "DAX"),
            ("^GSPC", "S&P500"),
            ("^DJI", "Dow Jones"),
            ("^VIX", "VIX"),
            ("GC=F", "Gold"),
            ("CL=F", "WTI Oil"),
            ("BZ=F", "Brent Oil"),
            ("BTC-USD", "Bitcoin")
        ]

        # Fetch each index with fallback (including FRED symbols for missing data)
        dax_hourly = self._fetch_index_with_fallback("^GDAXI", "DAX", period)
        snp500_hourly = self._fetch_index_with_fallback("^GSPC", "S&P500", period)
        dji_hourly = self._fetch_index_with_fallback("^DJI", "Dow Jones", period)
        vix = self._fetch_index_with_fallback("^VIX", "VIX", period)
        gold = self._fetch_index_with_fallback("GC=F", "Gold", period)
        crude_oil = self._fetch_index_with_fallback("CL=F", "WTI Oil", period)
        brent_oil = self._fetch_index_with_fallback("BZ=F", "Brent Oil", period)
        btc_usd = self._fetch_index_with_fallback("BTC-USD", "Bitcoin", period)

        # Prepare to merge
        dax_hourly_to_merge = self._get_growth_df(dax_hourly, 'dax')
        snp500_to_merge = self._get_growth_df(snp500_hourly, 'snp500')
        dji_hourly_to_merge = self._get_growth_df(dji_hourly, 'dji')
        vix_to_merge = self._get_growth_df(vix, 'vix')
        gold_to_merge = self._get_growth_df(gold, 'gold')
        crude_oil_to_merge = self._get_growth_df(crude_oil, 'wti_oil')
        brent_oil_to_merge = self._get_growth_df(brent_oil, 'brent_oil')
        btc_usd_to_merge = self._get_growth_df(btc_usd, 'btc_usd')

        # Merging
        m2 = pd.merge(snp500_to_merge,
                      dax_hourly_to_merge,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        m3 = pd.merge(m2,
                      dji_hourly_to_merge,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        m4 = pd.merge(m3,
                      vix_to_merge,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        m5 = pd.merge(m4,
                      gold_to_merge,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        m6 = pd.merge(m5,
                      crude_oil_to_merge,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        m7 = pd.merge(m6,
                      brent_oil_to_merge,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        m8 = pd.merge(m7,
                      btc_usd_to_merge,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        self.indexes_df = m8

    def fetch_macro(self, min_date=None):
        '''Fetch Macro data from FRED (using Pandas datareader)'''
        if min_date is None:
            min_date = "2023-08-27"
        else:
            min_date = pd.to_datetime(min_date)

        # Real Potential Gross Domestic Product (GDPPOT), Billions of Chained 2012 Dollars, QUARTERLY
        gdppot = pdr.DataReader("GDPPOT", "fred", start=min_date)
        gdppot['gdppot_us_yoy'] = gdppot.GDPPOT / gdppot.GDPPOT.shift(4) - 1
        gdppot['gdppot_us_qoq'] = gdppot.GDPPOT / gdppot.GDPPOT.shift(1) - 1
        time.sleep(1)

        cpilfesl = pdr.DataReader("CPILFESL", "fred", start=min_date)
        cpilfesl['cpi_core_yoy'] = cpilfesl.CPILFESL / cpilfesl.CPILFESL.shift(12) - 1
        cpilfesl['cpi_core_mom'] = cpilfesl.CPILFESL / cpilfesl.CPILFESL.shift(1) - 1
        time.sleep(1)

        trade_balance = pdr.DataReader("BOPGSTB", "fred", start=min_date)
        trade_balance['trade_balance_us_yoy'] = trade_balance.BOPGSTB / trade_balance.BOPGSTB.shift(12) - 1
        trade_balance['trade_balance_us_mom'] = trade_balance.BOPGSTB / trade_balance.BOPGSTB.shift(1) - 1

        fedfunds = pdr.DataReader("FEDFUNDS", "fred", start=min_date)
        time.sleep(1)

        dgs1 = pdr.DataReader("DGS1", "fred", start=min_date)
        time.sleep(1)

        dgs5 = pdr.DataReader("DGS5", "fred", start=min_date)
        time.sleep(1)

        dgs10 = pdr.DataReader("DGS10", "fred", start=min_date)
        time.sleep(1)

        gdppot_to_merge = gdppot[['gdppot_us_yoy', 'gdppot_us_qoq']]
        cpilfesl_to_merge = cpilfesl[['cpi_core_yoy', 'cpi_core_mom']]
        trade_balance_to_merge = trade_balance[['trade_balance_us_yoy', 'trade_balance_us_mom']]

        # Merging - start from daily stats (dgs1)
        m2 = pd.merge(dgs1,
                      dgs5,
                      left_index=True,
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        m2['Date'] = m2.index
        m2['Quarter'] = m2.Date.dt.to_period('Q').dt.to_timestamp()

        m3 = pd.merge(m2,
                      gdppot_to_merge,
                      left_on='Quarter',
                      right_index=True,
                      how='left',
                      validate='many_to_one')

        m3['Month'] = m2.Date.dt.to_period('M').dt.to_timestamp()

        m4 = pd.merge(m3,
                      cpilfesl_to_merge,
                      left_on='Month',
                      right_index=True,
                      how='left',
                      validate='many_to_one')

        m5 = pd.merge(m4,
                      trade_balance_to_merge,
                      how='left',
                      left_on='Month',
                      right_index=True,
                      validate="many_to_one")

        m6 = pd.merge(m5,
                      fedfunds,
                      left_on='Month',
                      right_index=True,
                      how='left',
                      validate='many_to_one')

        m7 = pd.merge(m6,
                      dgs10,
                      left_on='Date',
                      right_index=True,
                      how='left',
                      validate='one_to_one')

        fields_to_fill = ['cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS', 'DGS1', 'DGS5', 'DGS10']
        for field in fields_to_fill:
            m7[field] = m7[field].ffill()

        self.macro_df = m7

    def persist(self, data_dir: str):
        '''Save dataframes to files in a local directory 'dir' '''
        os.makedirs(data_dir, exist_ok=True)

        # Only save if dataframes exist and are not empty
        if self.ticker_df is not None and not self.ticker_df.empty:
            file_name = 'tickers_df.parquet'
            if os.path.exists(os.path.join(data_dir, file_name)):
                os.remove(os.path.join(data_dir, file_name))
            self.ticker_df.to_parquet(os.path.join(data_dir, file_name), compression='brotli')
            print(f"Saved {len(self.ticker_df)} ticker records")
        else:
            print("No ticker data to save")

        if self.indexes_df is not None and not self.indexes_df.empty:
            file_name = 'indexes_df.parquet'
            if os.path.exists(os.path.join(data_dir, file_name)):
                os.remove(os.path.join(data_dir, file_name))
            self.indexes_df.to_parquet(os.path.join(data_dir, file_name), compression='brotli')
            print(f"Saved {len(self.indexes_df)} index records")
        else:
            print("No index data to save")

        if self.macro_df is not None and not self.macro_df.empty:
            file_name = 'macro_df.parquet'
            if os.path.exists(os.path.join(data_dir, file_name)):
                os.remove(os.path.join(data_dir, file_name))
            self.macro_df.to_parquet(os.path.join(data_dir, file_name), compression='brotli')
            print(f"Saved {len(self.macro_df)} macro records")
        else:
            print("No macro data to save")

    def load(self, data_dir: str):
        """Load files from the local directory"""
        self.ticker_df = pd.read_parquet(os.path.join(data_dir, 'tickers_df.parquet'))
        self.macro_df = pd.read_parquet(os.path.join(data_dir, 'macro_df.parquet'))
        self.indexes_df = pd.read_parquet(os.path.join(data_dir, 'indexes_df.parquet'))



### 2. Transform


In [16]:
from typing import Optional

import pandas as pd
from tqdm import tqdm

import talib
import os



class TransformData:
    tickers_df: pd.DataFrame
    transformed_df: Optional[pd.DataFrame]
    def __init__(self, repo: DataRepository):
        # copy initial dfs from repo
        self.tickers_df = repo.ticker_df.copy(deep=True)
        self.macro_df = repo.macro_df.copy(deep=True)
        self.indexes_df = repo.indexes_df.copy(deep=True)

        # init transformed_df
        self.transformed_df = None

    def transform(self):
        '''Transform all dataframes from repo'''

        # Transform initial tickers_df to one with Tech indicators
        self._transform_tickers()

        # merge tickers (tech.indicators) with macro_df and indexes_df
        self._merge_tickers_macro_indexes_df()

        # truncate all data before 2000
        self.transformed_df = self.transformed_df[pd.to_datetime(self.transformed_df.Date) >= '2023-08-27']

    def _transform_tickers(self):
        '''Transform tickers dataframes from repo'''

        # TaLib needs inputs of a datatype 'Double'
        self.tickers_df['Volume'] = self.tickers_df['Volume'] * 1.0

        # Check which columns are available (Stooq doesn't have 'Adj Close')
        available_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        for key in available_columns:
            if key in self.tickers_df.columns:
                self.tickers_df.loc[:, key] = self.tickers_df[key].astype('double')

        merged_df = None

        # supress warnings
        pd.options.mode.chained_assignment = None  # default='warn'

        tickers = tqdm(self.tickers_df.Ticker.unique())
        for ticker in tickers:
            tickers.set_description(ticker)
            filter = (self.tickers_df.Ticker == ticker)
            current_ticker_df = self.tickers_df[filter]
            df_current_ticker_momentum_indicators = self._get_talib_momentum_indicators(current_ticker_df)
            df_current_ticker_volume_indicators = self._get_talib_volatility_cycle_price_indicators(current_ticker_df)
            df_current_ticker_pattern_indicators = self._get_talib_pattern_indicators(current_ticker_df)

            # need to have same 'utc' time on both sides of merges
            # https://stackoverflow.com/questions/73964894/you-are-trying-to-merge-on-datetime64ns-utc-and-datetime64ns-columns-if-yo
            current_ticker_df['DateTime'] = pd.to_datetime(current_ticker_df['DateTime'], utc=True)
            df_current_ticker_momentum_indicators['DateTime'] = pd.to_datetime(
                df_current_ticker_momentum_indicators['DateTime'], utc=True)
            df_current_ticker_volume_indicators['DateTime'] = pd.to_datetime(df_current_ticker_volume_indicators['DateTime'],
                                                                         utc=True)
            df_current_ticker_pattern_indicators['DateTime'] = pd.to_datetime(df_current_ticker_pattern_indicators['DateTime'],
                                                                          utc=True)

            # drop duplicates if any
            df_current_ticker_momentum_indicators = df_current_ticker_momentum_indicators.drop_duplicates(
                subset=["DateTime", "Ticker"])
            df_current_ticker_volume_indicators = df_current_ticker_volume_indicators.drop_duplicates(
                subset=["DateTime", "Ticker"])
            df_current_ticker_pattern_indicators = df_current_ticker_pattern_indicators.drop_duplicates(
                subset=["DateTime", "Ticker"])

            # Ensure all DataFrames have clean indexes without Date conflicts
            df_current_ticker_momentum_indicators = df_current_ticker_momentum_indicators.reset_index(drop=True)
            df_current_ticker_volume_indicators = df_current_ticker_volume_indicators.reset_index(drop=True)
            df_current_ticker_pattern_indicators = df_current_ticker_pattern_indicators.reset_index(drop=True)
            current_ticker_df = current_ticker_df.reset_index(drop=True)

            # merge to one df
            m1 = pd.merge(current_ticker_df, df_current_ticker_momentum_indicators, how='left', on=["DateTime", "Ticker", "Date"],
                          validate="one_to_one")
            m2 = pd.merge(m1, df_current_ticker_volume_indicators, how='left', on=["DateTime", "Ticker", "Date"],
                          validate="one_to_one")
            m3 = pd.merge(m2, df_current_ticker_pattern_indicators, how='left', on=["DateTime", "Ticker", "Date"],
                          validate="one_to_one")
            # m3 = current_ticker_df

            if merged_df is None:
                merged_df = m3
            else:
                merged_df = pd.concat([merged_df, m3], ignore_index=False)

        self.transformed_df = merged_df

    def _get_talib_momentum_indicators(self, df: pd.DataFrame) -> pd.DataFrame:

        momentum_df = None
        # ADX - Average Directional Movement Index
        talib_momentum_adx = talib.ADX(df.High.values, df.Low.values, df.Close.values, timeperiod=14)
        # ADXR - Average Directional Movement Index Rating
        talib_momentum_adxr = talib.ADXR(df.High.values, df.Low.values, df.Close.values, timeperiod=14)
        # APO - Absolute Price Oscillator
        talib_momentum_apo = talib.APO(df.Close.values, fastperiod=12, slowperiod=26, matype=0)
        # AROON - Aroon
        talib_momentum_aroon = talib.AROON(df.High.values, df.Low.values, timeperiod=14)
        # talib_momentum_aroon[0].size
        # talib_momentum_aroon[1].size
        # AROONOSC - Aroon Oscillator
        talib_momentum_aroonosc = talib.AROONOSC(df.High.values, df.Low.values, timeperiod=14)
        # BOP - Balance of Power
        # https://school.stockcharts.com/doku.php?id=technical_indicators:balance_of_power
        # calculate open prices as shifted closed prices from the prev day
        # open = df.Last.shift(1)
        talib_momentum_bop = talib.BOP(df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CCI - Commodity Channel Index
        talib_momentum_cci = talib.CCI(df.High.values, df.Low.values, df.Close.values, timeperiod=14)
        # CMO - Chande Momentum Oscillator
        talib_momentum_cmo = talib.CMO(df.Close.values, timeperiod=14)
        # DX - Directional Movement Index
        talib_momentum_dx = talib.DX(df.High.values, df.Low.values, df.Close.values, timeperiod=14)
        # MACD - Moving Average Convergence/Divergence
        talib_momentum_macd, talib_momentum_macdsignal, talib_momentum_macdhist = talib.MACD(df.Close.values,
                                                                                             fastperiod=12, \
                                                                                             slowperiod=26,
                                                                                             signalperiod=9)
        # MACDEXT - MACD with controllable MA type
        talib_momentum_macd_ext, talib_momentum_macdsignal_ext, talib_momentum_macdhist_ext = talib.MACDEXT(
            df.Close.values, \
            fastperiod=12, \
            fastmatype=0, \
            slowperiod=26, \
            slowmatype=0, \
            signalperiod=9, \
            signalmatype=0)
        # MACDFIX - Moving Average Convergence/Divergence Fix 12/26
        talib_momentum_macd_fix, talib_momentum_macdsignal_fix, talib_momentum_macdhist_fix = talib.MACDFIX(
            df.Close.values, \
            signalperiod=9)
        # MFI - Money Flow Index
        talib_momentum_mfi = talib.MFI(df.High.values, df.Low.values, df.Close.values, df.Volume.values, timeperiod=14)
        # MINUS_DI - Minus Directional Indicator
        talib_momentum_minus_di = talib.MINUS_DM(df.High.values, df.Low.values, timeperiod=14)
        # MOM - Momentum
        talib_momentum_mom = talib.MOM(df.Close.values, timeperiod=10)
        # PLUS_DI - Plus Directional Indicator
        talib_momentum_plus_di = talib.PLUS_DI(df.High.values, df.Low.values, df.Close.values, timeperiod=14)
        # PLUS_DM - Plus Directional Movement
        talib_momentum_plus_dm = talib.PLUS_DM(df.High.values, df.Low.values, timeperiod=14)
        # PPO - Percentage Price Oscillator
        talib_momentum_ppo = talib.PPO(df.Close.values, fastperiod=12, slowperiod=26, matype=0)
        # ROC - Rate of change : ((price/prevPrice)-1)*100
        talib_momentum_roc = talib.ROC(df.Close.values, timeperiod=10)
        # ROCP - Rate of change Percentage: (price-prevPrice)/prevPrice
        talib_momentum_rocp = talib.ROCP(df.Close.values, timeperiod=10)
        # ROCR - Rate of change ratio: (price/prevPrice)
        talib_momentum_rocr = talib.ROCR(df.Close.values, timeperiod=10)
        # ROCR100 - Rate of change ratio 100 scale: (price/prevPrice)*100
        talib_momentum_rocr100 = talib.ROCR100(df.Close.values, timeperiod=10)
        # RSI - Relative Strength Index
        talib_momentum_rsi = talib.RSI(df.Close.values, timeperiod=14)
        # STOCH - Stochastic
        talib_momentum_slowk, talib_momentum_slowd = talib.STOCH(df.High.values, df.Low.values, df.Close.values, \
                                                                 fastk_period=5, slowk_period=3, slowk_matype=0,
                                                                 slowd_period=3, slowd_matype=0)
        # STOCHF - Stochastic Fast
        talib_momentum_fastk, talib_momentum_fastd = talib.STOCHF(df.High.values, df.Low.values, df.Close.values, \
                                                                  fastk_period=5, fastd_period=3, fastd_matype=0)
        # STOCHRSI - Stochastic Relative Strength Index
        talib_momentum_fastk_rsi, talib_momentum_fastd_rsi = talib.STOCHRSI(df.Close.values, timeperiod=14, \
                                                                            fastk_period=5, fastd_period=3,
                                                                            fastd_matype=0)
        # TRIX - 1-day Rate-Of-Change (ROC) of a Triple Smooth EMA
        talib_momentum_trix = talib.TRIX(df.Close.values, timeperiod=30)
        # ULTOSC - Ultimate Oscillator
        talib_momentum_ultosc = talib.ULTOSC(df.High.values, df.Low.values, df.Close.values, timeperiod1=7,
                                             timeperiod2=14, timeperiod3=28)
        # WILLR - Williams' %R
        talib_momentum_willr = talib.WILLR(df.High.values, df.Low.values, df.Close.values, timeperiod=14)

        momentum_df = pd.DataFrame(
            {
                # assume here multi-index <dateTime, ticker>
                # 'datetime': df.index.get_level_values(0),
                # 'ticker': df.index.get_level_values(1) ,

                # old way with separate columns
                'DateTime': df.DateTime.values,
                'Date': df.Date.values,
                'Ticker': df.Ticker,

                'adx': talib_momentum_adx,
                'adxr': talib_momentum_adxr,
                'apo': talib_momentum_apo,
                'aroon_1': talib_momentum_aroon[0],
                'aroon_2': talib_momentum_aroon[1],
                'aroonosc': talib_momentum_aroonosc,
                'bop': talib_momentum_bop,
                'cci': talib_momentum_cci,
                'cmo': talib_momentum_cmo,
                'dx': talib_momentum_dx,
                'macd': talib_momentum_macd,
                'macdsignal': talib_momentum_macdsignal,
                'macdhist': talib_momentum_macdhist,
                'macd_ext': talib_momentum_macd_ext,
                'macdsignal_ext': talib_momentum_macdsignal_ext,
                'macdhist_ext': talib_momentum_macdhist_ext,
                'macd_fix': talib_momentum_macd_fix,
                'macdsignal_fix': talib_momentum_macdsignal_fix,
                'macdhist_fix': talib_momentum_macdhist_fix,
                'mfi': talib_momentum_mfi,
                'minus_di': talib_momentum_minus_di,
                'mom': talib_momentum_mom,
                'plus_di': talib_momentum_plus_di,
                'dm': talib_momentum_plus_dm,
                'ppo': talib_momentum_ppo,
                'roc': talib_momentum_roc,
                'rocp': talib_momentum_rocp,
                'rocr': talib_momentum_rocr,
                'rocr100': talib_momentum_rocr100,
                'rsi': talib_momentum_rsi,
                'slowk': talib_momentum_slowk,
                'slowd': talib_momentum_slowd,
                'fastk': talib_momentum_fastk,
                'fastd': talib_momentum_fastd,
                'fastk_rsi': talib_momentum_fastk_rsi,
                'fastd_rsi': talib_momentum_fastd_rsi,
                'trix': talib_momentum_trix,
                'ultosc': talib_momentum_ultosc,
                'willr': talib_momentum_willr,
            }
        )
        return momentum_df

    def _get_talib_volatility_cycle_price_indicators(self, df: pd.DataFrame) -> pd.DataFrame:

        # TA-Lib Volume indicators
        # https://github.com/TA-Lib/ta-lib-python/blob/master/docs/func_groups/volume_indicators.md
        # AD - Chaikin A/D Line
        talib_ad = talib.AD(df.High.values, df.Low.values, df.Close.values, df.Volume.values)
        # ADOSC - Chaikin A/D Oscillator
        talib_adosc = talib.ADOSC(
            df.High.values, df.Low.values, df.Close.values, df.Volume.values, fastperiod=3, slowperiod=10)
        # OBV - On Balance Volume
        talib_obv = talib.OBV(
            df.Close.values, df.Volume.values)

        # TA-Lib Volatility indicators
        # https://github.com/TA-Lib/ta-lib-python/blob/master/docs/func_groups/volatility_indicators.md
        # ATR - Average True Range
        talib_atr = talib.ATR(
            df.High.values, df.Low.values, df.Close.values, timeperiod=14)
        # NATR - Normalized Average True Range
        talib_natr = talib.NATR(
            df.High.values, df.Low.values, df.Close.values, timeperiod=14)
        # OBV - On Balance Volume
        talib_obv = talib.OBV(
            df.Close.values, df.Volume.values)

        # TA-Lib Cycle Indicators
        # https://github.com/TA-Lib/ta-lib-python/blob/master/docs/func_groups/cycle_indicators.md
        # HT_DCPERIOD - Hilbert Transform - Dominant Cycle Period
        talib_ht_dcperiod = talib.HT_DCPERIOD(df.Close.values)
        # HT_DCPHASE - Hilbert Transform - Dominant Cycle Phase
        talib_ht_dcphase = talib.HT_DCPHASE(df.Close.values)
        # HT_PHASOR - Hilbert Transform - Phasor Components
        talib_ht_phasor_inphase, talib_ht_phasor_quadrature = talib.HT_PHASOR(
            df.Close.values)
        # HT_SINE - Hilbert Transform - SineWave
        talib_ht_sine_sine, talib_ht_sine_leadsine = talib.HT_SINE(
            df.Close.values)
        # HT_TRENDMODE - Hilbert Transform - Trend vs Cycle Mode
        talib_ht_trendmode = talib.HT_TRENDMODE(df.Close.values)

        # TA-Lib Price Transform Functions
        # https://github.com/TA-Lib/ta-lib-python/blob/master/docs/func_groups/price_transform.md
        # AVGPRICE - Average Price
        talib_avgprice = talib.AVGPRICE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # MEDPRICE - Median Price
        talib_medprice = talib.MEDPRICE(df.High.values, df.Low.values)
        # TYPPRICE - Typical Price
        talib_typprice = talib.TYPPRICE(
            df.High.values, df.Low.values, df.Close.values)
        # WCLPRICE - Weighted Close Price
        talib_wclprice = talib.WCLPRICE(
            df.High.values, df.Low.values, df.Close.values)

        volume_volatility_cycle_price_df = pd.DataFrame(
            {'DateTime': df.DateTime.values,
             'Date': df.Date.values,
             'Ticker': df.Ticker,
             # TA-Lib Volume indicators
             'ad': talib_ad,
             'adosc': talib_adosc,
             'obv': talib_obv,
             # TA-Lib Volatility indicators
             'atr': talib_atr,
             'natr': talib_natr,
             'obv': talib_obv,
             # TA-Lib Cycle Indicators
             'ht_dcperiod': talib_ht_dcperiod,
             'ht_dcphase': talib_ht_dcphase,
             'ht_phasor_inphase': talib_ht_phasor_inphase,
             'ht_phasor_quadrature': talib_ht_phasor_quadrature,
             'ht_sine_sine': talib_ht_sine_sine,
             'ht_sine_leadsine': talib_ht_sine_leadsine,
             'ht_trendmod': talib_ht_trendmode,
             # TA-Lib Price Transform Functions
             'avgprice': talib_avgprice,
             'medprice': talib_medprice,
             'typprice': talib_typprice,
             'wclprice': talib_wclprice,
             }
        )

        # Need a proper date type
        volume_volatility_cycle_price_df['DateTime'] = pd.to_datetime(
            volume_volatility_cycle_price_df['DateTime'], utc=True)

        return volume_volatility_cycle_price_df

    def _get_talib_pattern_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        # TA-Lib Pattern Recognition indicators
        # https://github.com/TA-Lib/ta-lib-python/blob/master/docs/func_groups/pattern_recognition.md
        # Nice article about candles (pattern recognition) https://medium.com/analytics-vidhya/recognizing-over-50-candlestick-patterns-with-python-4f02a1822cb5

        # CDL2CROWS - Two Crows
        talib_cdl2crows = talib.CDL2CROWS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDL3BLACKCROWS - Three Black Crows
        talib_cdl3blackrows = talib.CDL3BLACKCROWS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDL3INSIDE - Three Inside Up/Down
        talib_cdl3inside = talib.CDL3INSIDE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDL3LINESTRIKE - Three-Line Strike
        talib_cdl3linestrike = talib.CDL3LINESTRIKE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDL3OUTSIDE - Three Outside Up/Down
        talib_cdl3outside = talib.CDL3OUTSIDE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDL3STARSINSOUTH - Three Stars In The South
        talib_cdl3starsinsouth = talib.CDL3STARSINSOUTH(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDL3WHITESOLDIERS - Three Advancing White Soldiers
        talib_cdl3whitesoldiers = talib.CDL3WHITESOLDIERS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLABANDONEDBABY - Abandoned Baby
        talib_cdlabandonedbaby = talib.CDLABANDONEDBABY(
            df.Open.values, df.High.values, df.Low.values, df.Close.values, penetration=0)
        # CDLADVANCEBLOCK - Advance Block
        talib_cdladvancedblock = talib.CDLADVANCEBLOCK(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLBELTHOLD - Belt-hold
        talib_cdlbelthold = talib.CDLBELTHOLD(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLBREAKAWAY - Breakaway
        talib_cdlbreakaway = talib.CDLBREAKAWAY(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLCLOSINGMARUBOZU - Closing Marubozu
        talib_cdlclosingmarubozu = talib.CDLCLOSINGMARUBOZU(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLCONCEALBABYSWALL - Concealing Baby Swallow
        talib_cdlconcealbabyswall = talib.CDLCONCEALBABYSWALL(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLCOUNTERATTACK - Counterattack
        talib_cdlcounterattack = talib.CDLCOUNTERATTACK(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLDARKCLOUDCOVER - Dark Cloud Cover
        talib_cdldarkcloudcover = talib.CDLDARKCLOUDCOVER(
            df.Open.values, df.High.values, df.Low.values, df.Close.values, penetration=0)
        # CDLDOJI - Doji
        talib_cdldoji = talib.CDLDOJI(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLDOJISTAR - Doji Star
        talib_cdldojistar = talib.CDLDOJISTAR(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLDRAGONFLYDOJI - Dragonfly Doji
        talib_cdldragonflydoji = talib.CDLDRAGONFLYDOJI(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLENGULFING - Engulfing Pattern
        talib_cdlengulfing = talib.CDLENGULFING(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)

        # CDLEVENINGDOJISTAR - Evening Doji Star
        talib_cdleveningdojistar = talib.CDLEVENINGDOJISTAR(
            df.Open.values, df.High.values, df.Low.values, df.Close.values, penetration=0)
        # CDLEVENINGSTAR - Evening Star
        talib_cdleveningstar = talib.CDLEVENINGSTAR(
            df.Open.values, df.High.values, df.Low.values, df.Close.values, penetration=0)
        # CDLGAPSIDESIDEWHITE - Up/Down-gap side-by-side white lines
        talib_cdlgapsidesidewhite = talib.CDLGAPSIDESIDEWHITE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLGRAVESTONEDOJI - Gravestone Doji
        talib_cdlgravestonedoji = talib.CDLGRAVESTONEDOJI(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLHAMMER - Hammer
        talib_cdlhammer = talib.CDLHAMMER(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLHANGINGMAN - Hanging Man
        talib_cdlhangingman = talib.CDLHANGINGMAN(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLHARAMI - Harami Pattern
        talib_cdlharami = talib.CDLHARAMI(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLHARAMICROSS - Harami Cross Pattern
        talib_cdlharamicross = talib.CDLHARAMICROSS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLHIGHWAVE - High-Wave Candle
        talib_cdlhighwave = talib.CDLHIGHWAVE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLHIKKAKE - Hikkake Pattern
        talib_cdlhikkake = talib.CDLHIKKAKE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLHIKKAKEMOD - Modified Hikkake Pattern
        talib_cdlhikkakemod = talib.CDLHIKKAKEMOD(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)

        # CDLHOMINGPIGEON - Homing Pigeon
        talib_cdlhomingpigeon = talib.CDLHOMINGPIGEON(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLIDENTICAL3CROWS - Identical Three Crows
        talib_cdlidentical3crows = talib.CDLIDENTICAL3CROWS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLINNECK - In-Neck Pattern
        talib_cdlinneck = talib.CDLINNECK(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLINVERTEDHAMMER - Inverted Hammer
        talib_cdlinvertedhammer = talib.CDLINVERTEDHAMMER(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLKICKING - Kicking
        talib_cdlkicking = talib.CDLKICKING(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLKICKINGBYLENGTH - Kicking - bull/bear determined by the longer marubozu
        talib_cdlkickingbylength = talib.CDLKICKINGBYLENGTH(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLLADDERBOTTOM - Ladder Bottom
        talib_cdlladderbottom = talib.CDLLADDERBOTTOM(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLLONGLEGGEDDOJI - Long Legged Doji
        talib_cdllongleggeddoji = talib.CDLLONGLEGGEDDOJI(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLLONGLINE - Long Line Candle
        talib_cdllongline = talib.CDLLONGLINE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLMARUBOZU - Marubozu
        talib_cdlmarubozu = talib.CDLMARUBOZU(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLMATCHINGLOW - Matching Low
        talib_cdlmatchinglow = talib.CDLMATCHINGLOW(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)

        # CDLMATHOLD - Mat Hold
        talib_cdlmathold = talib.CDLMATHOLD(
            df.Open.values, df.High.values, df.Low.values, df.Close.values, penetration=0)
        # CDLMORNINGDOJISTAR - Morning Doji Star
        talib_cdlmorningdojistar = talib.CDLMORNINGDOJISTAR(
            df.Open.values, df.High.values, df.Low.values, df.Close.values, penetration=0)
        # CDLMORNINGSTAR - Morning Star
        talib_cdlmorningstar = talib.CDLMORNINGSTAR(
            df.Open.values, df.High.values, df.Low.values, df.Close.values, penetration=0)
        # CDLONNECK - On-Neck Pattern
        talib_cdlonneck = talib.CDLONNECK(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLPIERCING - Piercing Pattern
        talib_cdlpiercing = talib.CDLPIERCING(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLRICKSHAWMAN - Rickshaw Man
        talib_cdlrickshawman = talib.CDLRICKSHAWMAN(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLRISEFALL3METHODS - Rising/Falling Three Methods
        talib_cdlrisefall3methods = talib.CDLRISEFALL3METHODS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLSEPARATINGLINES - Separating Lines
        talib_cdlseparatinglines = talib.CDLSEPARATINGLINES(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLSHOOTINGSTAR - Shooting Star
        talib_cdlshootingstar = talib.CDLSHOOTINGSTAR(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLSHORTLINE - Short Line Candle
        talib_cdlshortline = talib.CDLSHORTLINE(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLSPINNINGTOP - Spinning Top
        talib_cdlspinningtop = talib.CDLSPINNINGTOP(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)

        # CDLSTALLEDPATTERN - Stalled Pattern
        talib_cdlstalledpattern = talib.CDLSTALLEDPATTERN(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLSTICKSANDWICH - Stick Sandwich
        talib_cdlsticksandwich = talib.CDLSTICKSANDWICH(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLTAKURI - Takuri (Dragonfly Doji with very long lower shadow)
        talib_cdltakuru = talib.CDLTAKURI(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLTASUKIGAP - Tasuki Gap
        talib_cdltasukigap = talib.CDLTASUKIGAP(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLTHRUSTING - Thrusting Pattern
        talib_cdlthrusting = talib.CDLTHRUSTING(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLTRISTAR - Tristar Pattern
        talib_cdltristar = talib.CDLTRISTAR(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLUNIQUE3RIVER - Unique 3 River
        talib_cdlunique3river = talib.CDLUNIQUE3RIVER(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLUPSIDEGAP2CROWS - Upside Gap Two Crows
        talib_cdlupsidegap2crows = talib.CDLUPSIDEGAP2CROWS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)
        # CDLXSIDEGAP3METHODS - Upside/Downside Gap Three Methods
        talib_cdlxsidegap3methods = talib.CDLXSIDEGAP3METHODS(
            df.Open.values, df.High.values, df.Low.values, df.Close.values)

        pattern_indicators_df = pd.DataFrame(
            {'DateTime': df.DateTime.values,
             'Date': df.Date.values,
             'Ticker': df.Ticker,
             # TA-Lib Pattern Recognition indicators
             'cdl2crows': talib_cdl2crows,
             'cdl3blackrows': talib_cdl3blackrows,
             'cdl3inside': talib_cdl3inside,
             'cdl3linestrike': talib_cdl3linestrike,
             'cdl3outside': talib_cdl3outside,
             'cdl3starsinsouth': talib_cdl3starsinsouth,
             'cdl3whitesoldiers': talib_cdl3whitesoldiers,
             'cdlabandonedbaby': talib_cdlabandonedbaby,
             'cdladvancedblock': talib_cdladvancedblock,
             'cdlbelthold': talib_cdlbelthold,
             'cdlbreakaway': talib_cdlbreakaway,
             'cdlclosingmarubozu': talib_cdlclosingmarubozu,
             'cdlconcealbabyswall': talib_cdlconcealbabyswall,
             'cdlcounterattack': talib_cdlcounterattack,
             'cdldarkcloudcover': talib_cdldarkcloudcover,
             'cdldoji': talib_cdldoji,
             'cdldojistar': talib_cdldojistar,
             'cdldragonflydoji': talib_cdldragonflydoji,
             'cdlengulfing': talib_cdlengulfing,
             'cdleveningdojistar': talib_cdleveningdojistar,
             'cdleveningstar': talib_cdleveningstar,
             'cdlgapsidesidewhite': talib_cdlgapsidesidewhite,
             'cdlgravestonedoji': talib_cdlgravestonedoji,
             'cdlhammer': talib_cdlhammer,
             'cdlhangingman': talib_cdlhangingman,
             'cdlharami': talib_cdlharami,
             'cdlharamicross': talib_cdlharamicross,
             'cdlhighwave': talib_cdlhighwave,
             'cdlhikkake': talib_cdlhikkake,
             'cdlhikkakemod': talib_cdlhikkakemod,
             'cdlhomingpigeon': talib_cdlhomingpigeon,
             'cdlidentical3crows': talib_cdlidentical3crows,
             'cdlinneck': talib_cdlinneck,
             'cdlinvertedhammer': talib_cdlinvertedhammer,
             'cdlkicking': talib_cdlkicking,
             'cdlkickingbylength': talib_cdlkickingbylength,
             'cdlladderbottom': talib_cdlladderbottom,
             'cdllongleggeddoji': talib_cdllongleggeddoji,
             'cdllongline': talib_cdllongline,
             'cdlmarubozu': talib_cdlmarubozu,
             'cdlmatchinglow': talib_cdlmatchinglow,
             'cdlmathold': talib_cdlmathold,
             'cdlmorningdojistar': talib_cdlmorningdojistar,
             'cdlmorningstar': talib_cdlmorningstar,
             'cdlonneck': talib_cdlonneck,
             'cdlpiercing': talib_cdlpiercing,
             'cdlrickshawman': talib_cdlrickshawman,
             'cdlrisefall3methods': talib_cdlrisefall3methods,
             'cdlseparatinglines': talib_cdlseparatinglines,
             'cdlshootingstar': talib_cdlshootingstar,
             'cdlshortline': talib_cdlshortline,
             'cdlspinningtop': talib_cdlspinningtop,
             'cdlstalledpattern': talib_cdlstalledpattern,
             'cdlsticksandwich': talib_cdlsticksandwich,
             'cdltakuru': talib_cdltakuru,
             'cdltasukigap': talib_cdltasukigap,
             'cdlthrusting': talib_cdlthrusting,
             'cdltristar': talib_cdltristar,
             'cdlunique3river': talib_cdlunique3river,
             'cdlupsidegap2crows': talib_cdlupsidegap2crows,
             'cdlxsidegap3methods': talib_cdlxsidegap3methods
             }
        )

        # Need a proper date type

        pattern_indicators_df['DateTime'] = pd.to_datetime(
            pattern_indicators_df['DateTime'], utc=True)

        return pattern_indicators_df

    def _merge_tickers_macro_indexes_df(self):
        if self.transformed_df is None:
            return

        # assuming non-None transformed_df
        self.macro_df['MacroDate'] = pd.to_datetime(self.macro_df['Date'], utc=True)
        self.indexes_df['DateTime'] = pd.to_datetime(self.indexes_df.index, utc=True)

        self.macro_df.set_index('MacroDate', inplace=True)
        self.indexes_df.set_index('DateTime', inplace=True)

        self.macro_df = self.macro_df.drop(columns=['Date'], errors='ignore')
        self.indexes_df = self.indexes_df.drop(columns=['Date'], errors='ignore')

        # Merge macro data on daily basis
        self.transformed_df['MacroJoinDate'] = pd.to_datetime(self.transformed_df['Date'], utc=True)
        self.transformed_df = pd.merge(self.transformed_df,
                                       self.macro_df,
                                       how='left',
                                       left_on='MacroJoinDate',
                                       right_index=True,
                                       validate="many_to_one"
                                       )

        # Merge index data on hourly basis
        self.transformed_df = pd.merge(self.transformed_df,
                                       self.indexes_df,
                                       how='left',
                                       left_on='DateTime',
                                       right_index=True,
                                       validate="many_to_one"
                                       )

    def persist(self, data_dir: str):
        '''Save dataframes to files in a local directory 'dir' '''
        os.makedirs(data_dir, exist_ok=True)

        file_name = 'transformed_df.parquet'
        if os.path.exists(file_name):
            os.remove(file_name)
        self.transformed_df.to_parquet(os.path.join(data_dir, file_name), compression='brotli')

    def load(self, data_dir: str):
        """Load files from the local directory"""
        os.makedirs(data_dir, exist_ok=True)
        self.transformed_df = pd.read_parquet(os.path.join(data_dir, 'transformed_df.parquet'))


### 3. Training


In [17]:
import pandas as pd
import numpy as np
import os
import joblib
import xgboost as xgb

xgb.set_config(verbosity=0)


class TrainModel:
    def __init__(self, transformed):
        # Correct dataframe from TransformData
        self.df_full = transformed.transformed_df.copy()
        self.df_full['ln_volume'] = self.df_full.Volume.apply(lambda x: np.log(x) if x > 0 else np.nan)

        self.models = {}  # will hold models for 1h and 4h
        self.prediction_cols = []

    # --------------------------
    # Feature sets
    # --------------------------
    def _define_feature_sets(self):
        self.GROWTH = [g for g in self.df_full if (g.find('growth_') == 0) & (g.find('future') < 0)]
        self.OHLCV = ['Open', 'High', 'Low', 'Close', 'Volume']
        self.CATEGORICAL = ['Month', 'Weekday', 'Ticker']
        self.TO_PREDICT = [g for g in self.df_full.keys() if (g.find('future') >= 0)]
        self.MACRO = [
            'gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS',
            'trade_balance_us_yoy', 'trade_balance_us_mom', 'DGS1', 'DGS5', 'DGS10'
        ]
        self.CUSTOM_NUMERICAL = [
            'SMA10', 'SMA20', 'growing_moving_average',
            'high_minus_low_relative', 'volatility', 'ln_volume'
        ]

        self.TO_DROP = ['Year', 'Date', 'index_x', 'index_y', 'index', 'Quarter'] + self.CATEGORICAL + self.OHLCV

        self.TECHNICAL_INDICATORS = [
            'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo',
            'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext',
            'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm',
            'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd',
            'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'ad', 'adosc', 'obv', 'atr', 'natr',
            'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature',
            'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice'
        ]
        self.TECHNICAL_PATTERNS = [g for g in self.df_full.keys() if g.find('cdl') >= 0]

        self.NUMERICAL = (
            self.GROWTH + self.TECHNICAL_INDICATORS + self.TECHNICAL_PATTERNS +
            self.CUSTOM_NUMERICAL + self.MACRO
        )

        self.OTHER = [k for k in self.df_full.keys()
                      if k not in self.OHLCV + self.CATEGORICAL + self.NUMERICAL +
                      self.TO_DROP + self.TO_PREDICT]

    def _define_dummies(self):
        self.df_full.loc[:, 'Month'] = pd.to_datetime(self.df_full.Month_x, errors='coerce').dt.strftime('%B')
        self.df_full['Weekday'] = self.df_full['Weekday'].astype(str)
        self.df_full['Month_Weekday'] = self.df_full['Month'] + '_' + self.df_full['Weekday']
        self.CATEGORICAL.append('Month_Weekday')

        dummy_variables = pd.get_dummies(self.df_full[self.CATEGORICAL], dtype='int32')
        self.df_full = pd.concat([self.df_full, dummy_variables], axis=1)
        self.DUMMIES = dummy_variables.keys().to_list()

    # --------------------------
    # Temporal split
    # --------------------------
    def _perform_temporal_split(self, df, min_date, max_date, train_prop=0.7, val_prop=0.15):
        train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
        val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)

        split_labels = []
        for date in df['Date']:
            if date <= train_end:
                split_labels.append('train')
            elif date <= val_end:
                split_labels.append('validation')
            else:
                split_labels.append('test')

        df['split'] = split_labels
        return df

    def _clean_dataframe_from_inf_and_nan(self, df):
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.fillna(0)
        return df

    # --------------------------
    # ML dataframes
    # --------------------------
    def _define_dataframes_for_ML(self):
        features_list = self.NUMERICAL + self.DUMMIES
        to_predict = ['is_positive_growth_1h_future', 'is_positive_growth_4h_future']

        self.train_df = self.df_full[self.df_full.split == 'train'].copy()
        self.valid_df = self.df_full[self.df_full.split == 'validation'].copy()
        self.train_valid_df = self.df_full[self.df_full.split.isin(['train', 'validation'])].copy()
        self.test_df = self.df_full[self.df_full.split == 'test'].copy()

        def split_xy(df):
            X = df[features_list].copy()
            y = df[to_predict].copy()
            return self._clean_dataframe_from_inf_and_nan(X), y

        self.X_train, self.y_train = split_xy(self.train_df)
        self.X_valid, self.y_valid = split_xy(self.valid_df)
        self.X_train_valid, self.y_train_valid = split_xy(self.train_valid_df)
        self.X_test, self.y_test = split_xy(self.test_df)
        self.X_all, self.y_all = split_xy(self.df_full)

        print(f'X_train {self.X_train.shape}, X_valid {self.X_valid.shape}, '
              f'X_test {self.X_test.shape}, X_all {self.X_all.shape}')

    def prepare_dataframe(self):
        print("Preparing dataframe...")
        self._define_feature_sets()
        self._define_dummies()
        min_date, max_date = self.df_full.Date.min(), self.df_full.Date.max()
        self._perform_temporal_split(self.df_full, min_date, max_date)
        self._define_dataframes_for_ML()

    # --------------------------
    # Training
    # --------------------------
    def train_xgboost(self):
        print("Training separate XGBoost models for 1h and 4h...")

        params = dict(
            colsample_bytree=0.762959801713856,
            gamma=0.5382156736511214,
            learning_rate=0.019405744787252217,
            max_depth=8,
            min_child_weight=4.873591268079564,
            n_estimators=150,
            reg_alpha=0.9168585284681378,
            reg_lambda=0.679502423771298,
            subsample=0.312094076585974,
            n_jobs=-1,
            eval_metric='auc',
            objective='binary:logistic',
            random_state=42
        )

        targets = {
            "1h": "is_positive_growth_1h_future",
            "4h": "is_positive_growth_4h_future"
        }

        for horizon, target in targets.items():
            print(f"→ Training {horizon} model...")
            model = xgb.XGBClassifier(**params)
            model.fit(self.X_train_valid, self.y_train_valid[target])
            self.models[horizon] = model

    def persist(self, data_dir: str):
        os.makedirs(data_dir, exist_ok=True)
        for horizon, model in self.models.items():
            model_filename = f"xgboost_model_{horizon}.joblib"
            joblib.dump(model, os.path.join(data_dir, model_filename))

    def load(self, data_dir: str):
        self.models = {}
        for horizon in ["1h", "4h"]:
            path = os.path.join(data_dir, f"xgboost_model_{horizon}.joblib")
            if os.path.exists(path):
                self.models[horizon] = joblib.load(path)

    # --------------------------
    # Inference
    # --------------------------
    def make_inference(self):
        for horizon, model in self.models.items():
            pred_col = f"pred_xgboost_{horizon}_best"
            prob_col = f"prob_pred_xgboost_{horizon}_best"
            rank_col = f"pred_xgboost_{horizon}_best_rank"

            y_pred_prob = model.predict_proba(self.X_all)[:, 1]
            y_pred_class = (y_pred_prob >= 0.56).astype(int)

            self.df_full[pred_col] = y_pred_class.astype(bool)
            self.df_full[prob_col] = y_pred_prob
            self.df_full[rank_col] = self.df_full.groupby("Date")[prob_col].rank(method="first", ascending=False)

            self.prediction_cols.extend([pred_col, prob_col, rank_col])

        out_path = os.path.join("local_data", "tickers_df.parquet")
        os.makedirs("local_data", exist_ok=True)
        self.df_full.to_parquet(out_path, compression="brotli")
        print(f"Saved predictions (1h & 4h) to {out_path}")



### 4. Main Pipeline


In [18]:

import pandas as pd
import warnings
import os
from datetime import datetime

def main():
    FETCH_REPO = True
    TRANSFORM_DATA = True
    TRAIN_MODEL = True

    data_dir = '/content/drive/MyDrive/Colab Notebooks/PTIT/Time series/BTL/local_data/'
    ticker_file = os.path.join(data_dir, 'tickers_df.parquet')

    # Step 1: Get data
    print('Step 1: Getting data from APIs or Load from disk')
    repo = DataRepository()
    if FETCH_REPO:
        try:
            repo.fetch()
            repo.persist(data_dir=data_dir)
        except Exception as e:
            print(f"Failed to fetch fresh data: {e}")
            if os.path.exists(ticker_file):
                repo.load(data_dir=data_dir)
            else:
                return
    else:
        if os.path.exists(ticker_file):
            repo.load(data_dir=data_dir)
        else:
            return

    # Step 2: Transform data
    print('Step 2: Transforming data into one dataframe')
    transformed = TransformData(repo=repo)
    if TRANSFORM_DATA:
        transformed.transform()
        transformed.persist(data_dir=data_dir)
    else:
        transformed.load(data_dir=data_dir)

    # Step 3: Train/Load Model
    print('Step 3: Training the model or loading from disk')
    warnings.filterwarnings("ignore")
    trained = TrainModel(transformed=transformed)

    if TRAIN_MODEL:
        trained.prepare_dataframe()
        trained.train_xgboost()  # trains both 1h and 4h
        trained.persist(data_dir=data_dir)
    else:
        trained.prepare_dataframe()
        trained.load(data_dir=data_dir)

    # Step 4: Make Inference
    print('Step 4: Making inference')
    trained.make_inference()

    # Show only buy signals (prediction >= 0.56)
    for horizon in ["1h", "4h"]:
        pred_col = f"pred_xgboost_{horizon}_best"
        prob_col = f"prob_pred_xgboost_{horizon}_best"

        print(f"\n=== {horizon.upper()} Buy Predictions (prob >= 0.6) ===")
        buy_signals = trained.df_full.loc[
            trained.df_full[pred_col] & (trained.df_full[prob_col] >= 0.60),
            ["Ticker", pred_col, prob_col]
        ]

        if buy_signals.empty:
            print("No buy signals for this horizon.")
        else:
            print(buy_signals.sort_values(by=prob_col, ascending=False).reset_index(drop=True))

    current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M")
    print(f"\nCurrent date and time: {current_datetime}")


if __name__ == "__main__":
    main()
    print("\nLaunching Streamlit UI...")
    os.system("streamlit run streamlit_app.py")





Step 1: Getting data from APIs or Load from disk
Fetching Tickers info from YFinance
Going download data for this tickers: ['EURUSD=X', 'JPY=X', 'GBPUSD=X', 'AUDUSD=X', 'NZDUSD=X', 'EURJPY=X', 'GBPJPY=X', 'EURGBP=X', 'EURCAD=X', 'EURSEK=X', 'EURCHF=X', 'EURHUF=X', 'CNY=X', 'HKD=X', 'SGD=X', 'INR=X', 'MXN=X', 'PHP=X', 'IDR=X', 'THB=X', 'MYR=X', 'ZAR=X', 'RUB=X']


EURUSD=X:   0%|          | 0/23 [00:00<?, ?it/s]

yfinance SUCCESS: Got 474 rows for EURUSD=X


JPY=X:   4%|▍         | 1/23 [00:01<00:23,  1.07s/it]   

yfinance SUCCESS: Got 474 rows for JPY=X


...

yfinance SUCCESS: Got 400 rows for RUB=X


RUB=X: 100%|██████████| 23/23 [00:24<00:00,  1.07s/it]


Fetching Indexes info from YFinance
yfinance SUCCESS: Got 180 rows for DAX
yfinance SUCCESS: Got 140 rows for S&P500
yfinance SUCCESS: Got 140 rows for Dow Jones
yfinance SUCCESS: Got 282 rows for VIX
yfinance SUCCESS: Got 460 rows for Gold
yfinance SUCCESS: Got 460 rows for WTI Oil
yfinance SUCCESS: Got 460 rows for Brent Oil
yfinance SUCCESS: Got 673 rows for Bitcoin
Fetching Macro info from FRED (Pandas_datareader)
Saved 10330 ticker records
Saved 140 index records
Saved 669 macro records
Step 2: Transforming data into one dataframe


RUB=X: 100%|██████████| 23/23 [00:00<00:00, 31.13it/s]


Step 3: Training the model or loading from disk
Preparing dataframe...
X_train (7274, 211), X_valid (493, 211), X_test (2563, 211), X_all (10330, 211)
Training separate XGBoost models for 1h and 4h...
→ Training 1h model...
→ Training 4h model...
Step 4: Making inference
Saved predictions (1h & 4h) to local_data/tickers_df.parquet

=== 1H Buy Predictions (prob >= 0.6) ===
       Ticker  pred_xgboost_1h_best  prob_pred_xgboost_1h_best
0       JPY=X                  True                   0.741471
1    GBPJPY=X                  True                   0.726793
2    GBPJPY=X                  True                   0.726691
3       JPY=X                  True                   0.718336
4    EURJPY=X                  True                   0.718223
..        ...                   ...                        ...
655  GBPJPY=X                  True                   0.600280
656  GBPUSD=X                  True                   0.600273
657     THB=X                  True                   0.60

### 5. Start Pipeline


In [19]:
# DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/PTIT/Time series/BTL/local_data/'
# os.makedirs(DATA_DIR, exist_ok=True)

# print('1. Downloading Data...')
# repo = DataRepository()
# repo.fetch(period='1d')
# repo.persist(data_dir=DATA_DIR)

# print('2. Transforming Data...')
# xf = TransformData(repo=repo)
# xf.transform()
# xf.persist(data_dir=DATA_DIR)

# print('3. Training Models...')
# tr = TrainModel(transformed=xf)
# tr.prepare_dataframe()
# tr.train_xgboost()
# tr.persist(data_dir=DATA_DIR)

# print('4. Running Inference...')
# tr.make_inference()

# print('\nSUCCESS! You can now run the streamlit dashboard.')
